# Семинар 9 - Сигналы и Inter Process Communication

Процессам иногда бывает необходима коммуникация друг с другом. Все такие способы называются инструментами IPC, вот их примеры:
* Разделяемая память (`MAP_SHARED`)
* пайпы (`pipe`)
* именовые пайпы (`mkfifo`)
* сигналы
* сокеты
* ...

Про память и пайпы мы говорили на прошлых семинарах, сегодня обсудим именовые пайпы и сигналы.

```c
#include <sys/types.h>
#include <sys/stat.h>

int mkfifo(const char *path, mode_t mode);
```

Этот сискол создает файл на диске, который будет работать как пайп. То есть один процесс должен открыть его на чтение, другой на запись, и они смогут начать общение.

In [1]:
!gcc snippets/fifo/fifo.c -o snippets/fifo/fifo.out
!cat snippets/fifo/fifo.c

#include <stdio.h>
#include <sys/stat.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>
#include <fcntl.h>
#include <string.h>

int main(int argc, char* argv[]) {
    mkfifo("./fifo_demo", 0644);
    pid_t pid = fork();

    if (pid == 0) {
        int fd = open("./fifo_demo", O_WRONLY);
        dup2(fd, 1);
        close(fd);
        execlp(argv[1], argv[1], NULL);
    } else {
        // parent will read
        int fd = open("./fifo_demo", O_RDONLY);
        dup2(fd, 0);
        close(fd);

        int total_read = 0;
        char buffer[4096];
        int current_read;
        while ((current_read = read(0, buffer, sizeof(buffer))) > 0) {
            total_read += current_read;
        }
        printf("%d\n", total_read);
        waitpid(pid, NULL, 0);
    }
}


In [3]:
!./snippets/fifo/fifo.out pwd 

46


In [9]:
!pwd  # вывод команды
!pwd | wc -m  # количество символов в выводе

/Users/k.afentev/CAOS/caos-2025/sem09-signals
      46


По принципу работы именованые пайпы не отличаются от пайпа. У обоих есть следующие особенности:

Пайп буферизован, размер буффера, как правило, `65K`. Запись происходит в буфер, если в нем не хватает места, то блокируемся. Если при этом читающий конец закрыт, `write` завершится с ошибкой `Broken pipe`. При чтении если буфер непуст, читаем из него, иначе блокируемся. Если пишущий конец закрыт, `read` вернёт 0.

Последний пункт очень важен: он подсказывает, что нужно не забывать закрывать концы каналов, иначе все может зависнуть

### Сигналы

Сигнал это короткое сообщение для межпроцессорного взаимодействия. Чаще всего используются для прерывания/остановки процессов. 

Есть еще схожее понятие прерывания, но мы сегодня не о нём:
> Прерывания генерируются железом (например MMU может генерировать Page Fault) и обрабатываются ядром. А сигналы генерируются ядром и обрабатываются процессами.

Список всех сигналов можно посмотреть в `man 7 signal`. Вот нескольких основных:

| Сигнал  | Сочетание клавиш | Значение                                     |
|---------|------------------|----------------------------------------------|
| SIGINT  | Ctrl-C           | Завершение процесса с терминала              |
| SIGSTOP | Ctrl-Z           | Перевод процесса в STOPPED                   |
| SIGTERM |                  | Просьба процессу завершиться                 |
| SIGKILL |                  | Моментальное завершение процесса             |
| SIGQUIT | Ctrl-\           | Моментальное завершение процесса с терминала |

При получении сигнала процесс может:
* Выполнить действие по-умолчанию (для всех сигналов кроме `SIGCHLD` и `SIGUSR` это завершение)
* Игнорировать сигнал
* Выполнить заданное действие

Обратите внимание, что Ctrl+Z не убивает процесс!

Отправка (любых) сигналов осуществляется системным вызовом `kill`.

<img src="media/termination_meme.png" alt="Meme about process termination in linux and windows" width="250" style="background-color:white;"/>

``
int kill(pid_t pid, int signum);
``

Если `pid=-1`, сигнал отправится всем процессам пользователя.

С помощью `alarm` можно отправить самому себе `SIGALARM` через несколько секунд.

```
unsigned int alarm(unsigned int seconds);
```

In [10]:
!gcc snippets/alarm/alarm_demo.c -o snippets/alarm/alarm_demo.out
!cat snippets/alarm/alarm_demo.c

#include <unistd.h>
#include <stdio.h>

int main() {
    alarm(3);
    pause();
    return 1;
}

In [11]:
!./snippets/alarm/alarm_demo.out

Современный способ устанавливать обработчик сигналов -- системный вызов `sigaction`.

**Обработку SIGSTOP и SIGKILL переопределить нельзя**

```c
int sigaction(int signum,
              const struct sigaction *restrict act,
              struct sigaction *oldact);
// В oldact запишется текущий обработчик, можно передать NULL
```

Основные поля `sigaction`:

* `sa_handler` - указатель на функцию-обработчик с одним аргументом `int signum`
* `sa_flags` - флаги. Самый полезный, `SA_RESTART` продолжает выполнение системных вызовов после исполнения обработчика. 
* `sa_sigaction` - указатель на обработчик с тремя аргументами `int signum, siginfo_t *info, void *context`. Чтобы использовать его, а не `sa_handler`, нужно проставить флаг `SA_SIGINFO`.
* `sa_mask` - маска заблокированных сигналов на время исполнения обработчика. Про маски см. далее.

Сигнал может прийти процессу в любой момент. при этом исполнение текущего кода будет прервано и будет запущен обработчик сигнала. Поэтому актуальна проблема "гонок". В обработчиках нужно использовать специальный тип `sig_atomic_t`, он "атомарен относительно обработки сигналов" при операциях чтения и записи. Также может понадобиться добавить использовать `volatile`, чтобы запретить некоторые типы оптимизаций. Например, если есть цикл и флаг выставляется в обработчике сигнала, компилятор может соптимизовать его в `while (true)`.

```c
int quit = 0;
while (!quit) {
  ...
}
```

Также нельзя использовать не потокобезопасные функции такие как `printf` (список безопасных есть в `man 7 signal-safety`).

Обработку сигнала может прервать другой сигнал. Поэтому нужно делать максимально простые обработчики. Например, выставить флаг типа `sig_atomic_t` и как-то обработать его в цикле в основном коде

In [12]:
!gcc snippets/sigaction/sigaction_demo.c -o snippets/sigaction/sigaction_demo.out
!cat snippets/sigaction/sigaction_demo.c

#include <signal.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

volatile sig_atomic_t received_sigint = 0;

void sigint_handler(int signum) {
    received_sigint = 1;
}

void set_handler(int signum, void* handler, struct sigaction* action) {
    action->sa_handler = handler;
    action->sa_flags = SA_RESTART;
    sigaction(signum, action, NULL);
}

int main() {
    struct sigaction sigint_action = {0};
    set_handler(SIGINT, sigint_handler, &sigint_action);

    printf("%d\n", getpid());
    fflush(stdout);

    while (1) {
        pause();
        if (received_sigint) {
            printf("%s\n", "SIGINT received");
            fflush(stdout);
            received_sigint = 0;
        }
    }
    return 0;
}


In [13]:
!./snippets/sigaction/sigaction_demo.out 

40777
^C
SIGINT received


Сигналы, ожидающие доставки, представляются в виде маски т. е. их количество не учитывается. Маска своя для каждого потока. Рекомендуемые паттерн в многопоточных программах: выделить под обработку сигналов отдельный поток, а в остальных заблокировать их доставку через `pthread_sigmask`.

Есть и вторая маска -- маска заблокированных сигналов. Если придёт заблокированный сигнал, он будет обработан только после того, как пропадёт из маски заблокированнных.


Маски  сигналов описываются типом данных `sigset_t`.

Операции над маской:
 * `sigemptyset(sigset_t *set)` - инициализировать пустое множество;
 * `sigfillset(sigset_t *set)` - инициализировать полное множество;
 * `sigaddset(sigset_t *set, int signum)` - добавить сигнал к множеству;
 * `sigdelset(sigset_t *set, int signum)` - убрать сигнал из множества;
 * `sigismember(sigset_t *set, int signum)` - проверить наличие сигнала в множестве.

Можно поменять маску заблокированных сигналов с помощью `sigprocmask`:

```
int sigprocmask(int how, sigset_t *set, sigset_t *old_set);
```

Параметр `how` - это одно из значений:
 * `SIG_SETMASK` - установить множество сигналов в качестве маски блокируемых сигналов;
 * `SIG_BLOCK` - добавить множество к маске блокируемых сигналов;
 * `SIG_UNBLOCK` - убрать множество из маски блокируемых сигналов.

In [17]:
!gcc snippets/masks/sigprocmask_demo.c -o snippets/masks/sigprocmask_demo.out
!cat snippets/masks/sigprocmask_demo.c

#include <signal.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

volatile sig_atomic_t received_sigint = 0;

void sigint_handler(int signum) {
    received_sigint = 1;
}

void set_handler(int signum, void* handler, struct sigaction* action) {
    action->sa_handler = handler;
    action->sa_flags = SA_RESTART;
    sigaction(signum, action, NULL);
}

int main() {
    struct sigaction sigint_action;
    memset(&sigint_action, 0, sizeof(sigint_action));
    set_handler(SIGINT, sigint_handler, &sigint_action);

    printf("%d\n", getpid());
    fflush(stdout);

    sigset_t mask;
    sigemptyset(&mask);
    sigaddset(&mask, SIGINT);

    while (1) {
        sigprocmask(SIG_BLOCK, &mask, NULL);
        sleep(5);
        sigprocmask(SIG_UNBLOCK, &mask, NULL);
        if (received_sigint) {
            printf("%s\n", "SIGINT received");
            fflush(stdout);
            received_sigint = 0;
        }
    }
    return 0;
}


In [16]:
!./snippets/masks/sigprocmask_demo.out 

41811
^C


Системный вызов `pause()` позволяет приостановить выполнение до прихода любого незаблокированного сигнала.

Системный вызов `sigsuspend(sigset_t *temp_mask)` временно приостанавливает работу программы до тех пор, пока не придёт один из сигналов, отсутсвующий в множестве `temp_mask`. Сигналы, отсутсвующие в новом временном множестве, будут доставлены даже в том случае, если они ранее были заблокированы.

In [18]:
!gcc snippets/masks/sigsuspend_demo.c -o snippets/masks/sigsuspend_demo.out
!cat snippets/masks/sigsuspend_demo.c

#include <signal.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

volatile sig_atomic_t received_sigint = 0;

void sigint_handler(int signum) {
    received_sigint = 1;
}

void set_handler(int signum, void* handler, struct sigaction* action) {
    action->sa_handler = handler;
    action->sa_flags = SA_RESTART;
    sigaction(signum, action, NULL);
}

int main() {
    struct sigaction sigint_action;
    memset(&sigint_action, 0, sizeof(sigint_action));
    set_handler(SIGINT, sigint_handler, &sigint_action);

    printf("%d\n", getpid());
    fflush(stdout);

    sigset_t mask_without_sigint;
    sigfillset(&mask_without_sigint);
    sigdelset(&mask_without_sigint, SIGINT);

    while (1) {
        sigsuspend(&mask_without_sigint);
        if (received_sigint) {
            printf("%s\n", "SIGINT received");
            fflush(stdout);
            received_sigint = 0;
        }
    }
    return 0;
}


In [19]:
!./snippets/masks/sigsuspend_demo.out 

46656
^C
SIGINT received


В качестве альтернативы, можно использовать `int sigwaitinfo(const sigset_t *set, siginfo_t *info)`. Он приостанавливает программу до того, как придёт сигнал из маски (его нужно предварительно заблокировать через `sigprocmask`).  Функция возвращает номер сигнала, более подробную информацию можно получить из структуры.

In [20]:
!gcc snippets/masks/sigwaitinfo_demo.c -o snippets/masks/sigwaitinfo_demo.out
!cat snippets/masks/sigwaitinfo_demo.c

snippets/masks/sigwaitinfo_demo.c:16:22: error: call to undeclared function 'sigwaitinfo'; ISO C99 and later do not support implicit function declarations [-Wimplicit-function-declaration]
   16 |         int signum = sigwaitinfo(&mask, &info);
      |                      ^
1 error generated.
#include <signal.h>
#include <stdio.h>
#include <string.h>
#include <unistd.h>

int main() {
    sigset_t mask;
    sigemptyset(&mask);
    sigaddset(&mask, SIGINT);

    sigprocmask(SIG_BLOCK, &mask, NULL);

    siginfo_t info;

    while (1) {
        int signum = sigwaitinfo(&mask, &info);
        printf("%s siginfo=%d, signum=%d\n", "Received signal", info.si_signo, signum);
        fflush(stdout);
    }
    
    return 0;
}


In [21]:
!./snippets/masks/sigwaitinfo_demo.c

zsh:1: permission denied: ./snippets/masks/sigwaitinfo_demo.c


У рассмотренных сигналов есть множество проблем: неудобство асинхронной работы, отсутствие понятных гарантий и т.д. Чтобы частично улучшить эту ситуацию, в POSIX есть другой способ работы с сигналами: сигналы реального времени.

Сигналы раннего времени имеют номера от `SIGRTMIN` до `SIGRTMAX` и доставляются через очередь, поэтому учитывается их количество и порядок прихода. Также есть одно дополнительное поле для передачи целочисленной информации.


Для использования очереди сигналов, необходимо отправлять их с помощью функции `sigqueue`:

```c
#include <signal.h>
union sigval {
    int    sival_int;
    void*  sival_ptr;
};
int sigqueue(pid_t pid, int signum, const union sigval value);
```

Эта функция может завершиться с ошибкой `EAGAIN` в том случае, если исчерпан лимит на количество сигналов в очереди. Опциональное значение, передаваемое в качестве третьего параметра, может быть извлечено получателем из поля `si_value` структуры `siginfo_t`, если использовать вариант обработчика `sigaction` с тремя аргументами


In [22]:
!gcc snippets/realtime/sigqueue_demo.c -o snippets/realtime/sigqueue_demo.out
!cat snippets/realtime/sigqueue_demo.c

snippets/realtime/sigqueue_demo.c:14:24: error: no member named 'si_int' in 'struct __siginfo'
   14 |     last_value = info->si_int;
      |                  ~~~~  ^
snippets/realtime/sigqueue_demo.c:19:18: error: use of undeclared identifier 'SIGRTMIN'
   19 |     for (int i = SIGRTMIN; i <= SIGRTMAX; i++) {
      |                  ^
snippets/realtime/sigqueue_demo.c:19:33: error: use of undeclared identifier 'SIGRTMAX'
   19 |     for (int i = SIGRTMIN; i <= SIGRTMAX; i++) {
      |                                 ^
snippets/realtime/sigqueue_demo.c:33:22: error: use of undeclared identifier 'SIGRTMIN'
   33 |         for (int i = SIGRTMIN; i <= SIGRTMIN + 5; i++) {
      |                      ^
snippets/realtime/sigqueue_demo.c:33:37: error: use of undeclared identifier 'SIGRTMIN'
   33 |         for (int i = SIGRTMIN; i <= SIGRTMIN + 5; i++) {
      |                                     ^
snippets/realtime/sigqueue_demo.c:34:13: error: call to undeclared function 'sigqueue'; ISO

In [23]:
!./snippets/realtime/sigqueue_demo.c

zsh:1: permission denied: ./snippets/realtime/sigqueue_demo.c


### signalfd 
В Linux есть способ обрабатывать сигналы через специальный файловый дескриптор.

```c
int signalfd(int fd, const sigset_t *mask, int flags);
// fd позволяет заменить существующий файловый дескриптор (-1 если хотим просто открыть новый)
// маска отвечает за сигналы, которые мы хотим получать
// нужно их предварительно заблокировать через sigprocmask
// чтобы отключить обработку по-умолчанию
```

После этого можно читать из возвращённого файлового дескриптора структуру `signalfd_siginfo`. Список всех полей можно посмотреть в `man 2 signalfd`. Наиболее полезные:
* `ssi_signo` - номер сигнала
* `ssi_pid` - PID отправителя
* `ssi_int` - переданное число (для сигналов реального времени)

**Ограничения**: нельзя обрабатывать сигналы, которые генерируются синхронно, такие как `SIGSEGV` и `SIGFPE`.

In [24]:
!gcc snippets/signalfd/signalfd_demo.c -o snippets/signalfd/signalfd_demo.out
!cat snippets/signalfd/signalfd_demo.c

snippets/signalfd/signalfd_demo.c:4:10: fatal error: 'sys/signalfd.h' file not found
    4 | #include <sys/signalfd.h>
      |          ^~~~~~~~~~~~~~~~
1 error generated.
#include <signal.h>
#include <stdbool.h>
#include <stdio.h>
#include <sys/signalfd.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main() {
    sigset_t mask;
    sigfillset(&mask);
    sigdelset(&mask, SIGCONT);
    sigprocmask(SIG_BLOCK, &mask, NULL);

    printf("%d\n", getpid());
    fflush(stdout);

    int fd = signalfd(-1, &mask, 0);
    struct signalfd_siginfo fdsi;
    while (true) {
        read(fd, &fdsi, sizeof(struct signalfd_siginfo));
        printf("Got signal %d\n", fdsi.ssi_signo);
    }
    close(fd);
    return 0;
}


In [25]:
!./snippets/signalfd/signalfd_demo.out

zsh:1: no such file or directory: ./snippets/signalfd/signalfd_demo.out


> Можно при помощи пайпов реализовать свой платформо-независимый аналог `signal_fd`. 
Про это почитать можно [тут](https://github.com/yuri-pechatnov/caos/tree/master/caos_2020-2021/sem15-signal#-signalfd-%D0%B4%D0%BB%D1%8F-%D0%B1%D0%B5%D0%B4%D0%BD%D1%8B%D1%85).